In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
import numpy as np

dataset = pd.read_csv('https://raw.githubusercontent.com/Dataprof/ml-model-deployment/main/storepurchasedata_large.csv')
X=dataset.iloc[:, :-1].values
y=dataset.iloc[:,-1].values
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =.20,random_state=0)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
Xtrain_ = torch.from_numpy(X_train).type(torch.FloatTensor)
Xtest_ = torch.from_numpy(X_test).type(torch.FloatTensor)
ytrain_ = torch.from_numpy(y_train).type(torch.FloatTensor)
ytest_ = torch.from_numpy(y_test).type(torch.FloatTensor)
Xtrain_.shape, ytrain_.shape
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, hidden_size)
    self.fc3 = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    X = torch.relu((self.fc1(X)))
    X = torch.relu((self.fc2(X)))
    X = self.fc3(X)

    return F.log_softmax(X, dim=1)

model = Net()

import torch.optim as optim
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.NLLLoss()

epochs = 100
for epoch in range(epochs):
  optimizer.zero_grad()
  Ypred = model(Xtrain_)
  loss = loss_fn(Ypred, ytrain_.long())
  loss.backward()
  optimizer.step()
  print('Epoch', epoch, 'loss', loss.item())
list(model.parameters())

input_in_tensor= torch.from_numpy(sc.transform(np.array([[20,40000]]))).float()
y_cust_20_40000 = model(input_in_tensor)
y_cust_20_40000
predicted_cust_20_40000 = torch.max(y_cust_20_40000.data, 1)[1]
predicted_cust_20_40000

# Save the model - type1
torch.save(model,'customer_buy.pt')
# calling the model
restored_model = torch.load('customer_buy.pt', weights_only=False)
# Save the model - type2 recommended
torch.save(model.state_dict(), 'customer_buy_state_dict')
#calling the model
dict_model = torch.load('customer_buy_state_dict', weights_only=False)
new_predictor = Net()
new_predictor.load_state_dict(torch.load('customer_buy_state_dict'))